# Multiple Input and Multiple Output Channels
:label:`sec_channels`

- So far, we’ve mostly worked with **a single input and output channel**:
  - This allowed us to treat inputs, kernels, and outputs as **2D tensors**.

- However, real-world inputs often have **multiple channels**:
  - For example, **RGB images** have 3 channels: **Red**, **Green**, and **Blue**.
  - An RGB image has shape: $3 \times h \times w$.

- We call this size-3 axis the **channel dimension**.

- Once channels are included, both **inputs** and **hidden representations** become **3D tensors**.

- The concept of channels has been there since early CNNs:
  - E.g., **LeNet-5** :cite:`LeCun.Jackel.Bottou.ea.1995` used multiple channels.

- In this section, we will explore:
  - **Convolution kernels** with **multiple input** and **multiple output channels**.


In [1]:
import torch
from d2l import torch as d2l

## Multiple Input Channels

- When the **input data has multiple channels**, the **convolution kernel must match** that number of channels:
  - The kernel must have the **same number of input channels** as the input.

- Let:
  - $c_\textrm{i}$ be the **number of input channels**,
  - $k_\textrm{h} \times k_\textrm{w}$ be the **kernel’s height and width**.

- Then, the convolution kernel must have:
  - **$c_\textrm{i}$ input channels**, each with a **$k_\textrm{h} \times k_\textrm{w}$** matrix.

- Special case:
  - If $c_\textrm{i} = 1$, the kernel is simply a **2D tensor** of shape $k_\textrm{h} \times k_\textrm{w}$.


- When $c_\textrm{i} > 1$, the convolution kernel must include a **$k_\textrm{h} \times k_\textrm{w}$ tensor for each input channel**.

- By **concatenating** these $c_\textrm{i}$ tensors, the **kernel** becomes a **3D tensor** of shape:

  $$
  c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}
  $$

- **Cross-correlation procedure** for multi-channel input:
  1. For each **input channel** and corresponding **kernel slice**, perform **2D cross-correlation**.
  2. **Sum** all $c_\textrm{i}$ results **elementwise** to produce a **single 2D output tensor**.

- Final result:
  - A **2D tensor** representing the cross-correlation of:
    - A **multi-channel input**, and
    - A **multi-input-channel convolution kernel**.


- :numref:`fig_conv_multi_in` illustrates a **2D cross-correlation with two input channels**.

- The **shaded regions** indicate:
  - The **first output element**,
  - The **corresponding elements** of the input and kernel tensors used for the computation.

- Output value computation:
  $$
  (1\times1 + 2\times2 + 4\times3 + 5\times4) + (0\times0 + 1\times1 + 3\times2 + 4\times3) = 56
  $$

![Cross-correlation computation with two input channels.](../img/conv-multi-in.svg)
:label:`fig_conv_multi_in`

- To reinforce our understanding, we can:
  - **Implement cross-correlation with multiple input channels manually**.

- Key steps:
  - For each **input channel**, perform a **2D cross-correlation** with its corresponding **kernel slice**.
  - Then, **sum** all the resulting 2D outputs **elementwise**.


In [2]:
def corr2d_multi_in(X, K):
    # Iterate through the 0th dimension (channel) of K first, then add them up
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

- To validate the output shown in :numref:`fig_conv_multi_in`, we can construct **input tensor `X`** and corresponding **kernel tensor `K`**.
- The final result should match the **computed value**, as shown in the figure.


In [3]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

## Multiple Output Channels
:label:`subsec_multi-output-channels`

- Until now, we always ended up with **one output channel** regardless of how many input channels we had.

- But as discussed in :numref:`subsec_why-conv-channels`, it's essential to have **multiple output channels per layer**:
  - This allows the network to **learn richer and more diverse features**.

- In most neural network architectures:
  - The number of **channels increases** as we go deeper,
  - While the **spatial resolution decreases** (downsampling),
  - Creating a trade-off: fewer spatial positions, but **greater feature abstraction**.

- **Channel-wise intuition**:
  - You might imagine each output channel detects **different feature types** (e.g., edges, textures),
  - But in practice, channels are **jointly optimized**, not learned independently.

  - So:
    - A single channel might not strictly correspond to one visual feature,
    - Instead, **some direction in channel space** may represent a certain semantic concept.

- Notation:
  - Let:
    - $c_\textrm{i}$ = number of **input channels**,
    - $c_\textrm{o}$ = number of **output channels**,
    - $k_\textrm{h} \times k_\textrm{w}$ = height and width of each kernel.

- To produce multiple output channels:
  - For **each output channel**, we define a kernel tensor of shape $c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}$.
  - Stack these over the output channel dimension:
    - Resulting kernel shape:
      $$
      c_\textrm{o} \times c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}
      $$

- During cross-correlation:
  - Each output channel is computed by:
    - Taking all input channels,
    - Applying its **corresponding 3D kernel tensor**,
    - And **summing the results** across input channels.

- The function below implements cross-correlation to **produce multi-channel output**.


In [4]:
def corr2d_multi_in_out(X, K):
    # Iterate through the 0th dimension of K, and each time, perform
    # cross-correlation operations with input X. All of the results are
    # stacked together
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

- We construct a trivial convolution kernel with three output channels by concatenating the kernel tensor for `K` with `K+1` and `K+2`.


In [5]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

- Below, we perform cross-correlation operations on the **input tensor `X`** with the **kernel tensor `K`**.
  - Now the output contains **three channels**.
  - The result of the first channel is consistent with the result of the previous input tensor `X` and the multi-input channel, single-output channel kernel.


In [6]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

## $1\times 1$ Convolutional Layer
:label:`subsec_1x1`

- A **$1 \times 1$ convolution** (i.e., $k_\textrm{h} = k_\textrm{w} = 1$) might seem **counterintuitive**:
  - Regular convolutions are used to detect **local spatial patterns** across adjacent pixels.
  - A $1 \times 1$ kernel has **no spatial extent** — so it **cannot aggregate neighboring pixel information**.

- Despite this, **$1 \times 1$ convolutions are widely used** in modern architectures:
  - E.g., in **Network in Network** :cite:`Lin.Chen.Yan.2013`
  - And **Inception networks** :cite:`Szegedy.Ioffe.Vanhoucke.ea.2017`

- What does a $1 \times 1$ convolution do?

  - Because it only spans **a single spatial location**, it:
    - **Loses the ability** to detect **spatial features**,
    - But performs **computation across the channel dimension** instead.

- Therefore:
  - The $1 \times 1$ convolution is effectively a **per-pixel fully connected layer** over the channels.
  - It **combines input channel values** at each spatial location to produce new feature representations.

- In summary:
  - A $1 \times 1$ convolution is not about spatial correlation,
  - It is used for **channel-wise feature transformation** or **dimensionality adjustment**.


- :numref:`fig_conv_1x1` demonstrates how a **$1 \times 1$ convolution** works with:
  - **3 input channels**, and
  - **2 output channels**.

![The cross-correlation computation uses the $1\times 1$ convolution kernel with three input channels and two output channels. The input and output have the same height and width.](../img/conv-1x1.svg)
:label:`fig_conv_1x1`

- Key observations:
  - **Input and output** have the **same height and width**.
  - Each **output element** is computed from a **linear combination of channel values at the same spatial position** in the input.

- Interpretation:
  - A $1 \times 1$ convolution is equivalent to:
    - A **fully connected layer** applied **independently at each pixel location**.
    - It transforms a **$c_\textrm{i}$-dimensional input vector** into a **$c_\textrm{o}$-dimensional output vector**.

- Parameter count:
  - The layer requires:
    $$
    c_\textrm{o} \times c_\textrm{i} \text{ weights} + \text{biases}
    $$

- Weight sharing:
  - Because it’s a convolution, the **same weights are used at all spatial locations** (i.e., weight sharing still holds).

- Role of nonlinearity:
  - $1 \times 1$ convolutions are often **followed by nonlinear activation functions**.
  - This ensures they **cannot be absorbed** into larger convolutions via linear algebra.

- Practical implementation:
  - A $1 \times 1$ convolution can be implemented via a **fully connected layer**.
  - But we must **reshape the input/output tensors** appropriately before and after the matrix multiplication.


In [7]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

- When performing $1\times 1$ convolutions, the above function is equivalent to the previously implemented cross-correlation function `corr2d_multi_in_out`.


In [8]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

## Discussion

- **Channels** give us the best of both worlds:
  - The **nonlinear expressiveness** of MLPs,
  - Combined with the **local feature extraction** power of convolutions.

- Key advantages of channels:
  - Enable CNNs to **detect multiple types of features simultaneously**,
    - e.g., edges, corners, textures, and higher-level patterns.
  - Provide a **flexible balance** between:
    - **Translation invariance** (from local convolutions), and
    - The need for **rich and diverse representations** (via depth in channels).

- However, **increased channel depth** comes with **computational cost**:

  - For an image of size $(h \times w)$ and kernel size $k \times k$, the cost is:
    $$
    \mathcal{O}(h \cdot w \cdot k^2)
    $$

  - With $c_\textrm{i}$ input and $c_\textrm{o}$ output channels:
    $$
    \mathcal{O}(h \cdot w \cdot k^2 \cdot c_\textrm{i} \cdot c_\textrm{o})
    $$

  - Example:
    - A $256 \times 256$ image,
    - $5 \times 5$ kernel,
    - $128$ input and output channels:
    - Results in **over 53 billion operations** (including multiplications and additions).

- Later in the course, we will see techniques that **reduce this cost**, such as:
  - **Block-diagonal constraints** on the convolution,
  - Leading to architectures like **ResNeXt** :cite:`Xie.Girshick.Dollar.ea.2017`.
